# 03 — Fine-tune YOLO11s on SportsMOT basketball

**Run this on an A100 runtime.** This is the only notebook that actually
needs the A100; data prep and evaluation run fine on a free T4.

What this notebook does, end to end:

1. Sanity-checks the converted dataset and that you got an A100.
2. Fine-tunes **YOLO11s** (initialized from COCO weights) on the SportsMOT
   basketball train split for ~30 epochs at imgsz=1280.
3. Evaluates final mAP on the basketball val split.
4. Runs detection + BoT-SORT tracking on a held-out val sequence and
   computes MOTA / IDF1 — the same metrics as the local pretrained
   baseline, so you get a direct before/after comparison.
5. Copies `best.pt` to your Drive so you can download it back to your
   local machine for inference.

### Compute budget

| Step | Time on A100 | A100-hours used |
|------|-------------:|----------------:|
| Training (~30 epochs) | ~1.5–2 h | ~2.0 |
| Validation + tracking eval | ~5 min | ~0.1 |
| **Total** | | **~2.1 h** |

That leaves ~8 A100-hours / month for ablations (e.g. trying YOLO11m, or
tweaking augmentation) and for the M2/M3 milestones.

### Prerequisite
- ✅ `notebooks/01_download_and_convert.ipynb` ran successfully → converted
  dataset on Drive at `MyDrive/stat_tracking/sportsmot_yolo/`.

## 1. Setup, Drive mount, GPU check

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_COLAB = 'google.colab' in sys.modules
print('On Colab:', ON_COLAB)

if ON_COLAB:
    REPO_URL = os.environ.get('STAT_TRACKING_REPO', 'https://github.com/shreyas0328/stat_tracking.git')
    REPO_DIR = '/content/stat_tracking'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/stat_tracking'
else:
    DRIVE_ROOT = os.path.abspath('data')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
DATASET_ROOT  = os.path.join(DRIVE_ROOT, 'sportsmot_yolo')
RAW_ROOT      = os.path.join(DRIVE_ROOT, 'sportsmot_raw')   # for the post-train tracking eval
WEIGHTS_DRIVE = os.path.join(DRIVE_ROOT, 'weights')
os.makedirs(WEIGHTS_DRIVE, exist_ok=True)
print('Dataset:', DATASET_ROOT)
print('Weights out:', WEIGHTS_DRIVE)

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU detected. Switch runtime to GPU.'
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f'Device: {name} ({vram} GB VRAM)')
if 'A100' not in name:
    print('⚠️  This notebook is sized for an A100. On a T4/V100 the run will be much slower.')
    print('   Consider dropping batch=24 below to 8, and imgsz=1280 to 960.')

In [ ]:
# Sanity-check the converted dataset.
ds = Path(DATASET_ROOT)
assert ds.exists(), f'No dataset at {ds}. Run notebook 01 first.'
for split in ('train', 'val'):
    n_imgs = len(list((ds / 'images' / split).glob('*.jpg')))
    n_lbls = len(list((ds / 'labels' / split).glob('*.txt')))
    n_nonempty = sum(1 for p in (ds / 'labels' / split).glob('*.txt') if p.stat().st_size > 0)
    print(f'  {split:>5}: {n_imgs:>6} images, {n_lbls:>6} labels ({n_nonempty} non-empty)')
    assert n_imgs > 0, f'{split} is empty'
    assert n_nonempty > 0.5 * n_imgs, f'{split}: <50% of frames have labels — re-check notebook 01'

## 2. Write the dataset YAML for Ultralytics

In [ ]:
config_path = Path('configs/yolo11_sportsmot_colab.yaml')
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(f"""# auto-generated by 03_train_yolo.ipynb
path: {ds.resolve()}
train: images/train
val:   images/val
names:
  0: player
""")
print(config_path.read_text())

## 3. Train

Hyperparameters chosen for the A100 + this dataset size:

| Param | Value | Why |
|-------|-------|-----|
| `model` | `yolo11s.pt` | Best size/accuracy trade-off in our compute budget |
| `epochs` | 30 | Past ~25 epochs returns diminish on this dataset |
| `imgsz` | 1280 | Broadcast frames are 720p–1080p; 1280 keeps small/distant players resolvable |
| `batch` | 24 | Fits comfortably in A100 80GB at 1280 |
| `optimizer` | `AdamW` | More forgiving than SGD for short runs |
| `lr0` | 0.001 | Small LR for fine-tune (we're starting from COCO, not random init) |
| `close_mosaic` | 10 | Last 10 epochs run without mosaic for cleaner final weights |
| `patience` | 15 | Early stop if val mAP plateaus for 15 epochs |
| `save_period` | 5 | Checkpoint every 5 epochs in case Colab disconnects mid-run |

If the runtime disconnects, you can restart this cell and Ultralytics
will resume from `last.pt` automatically when `resume=True` is set.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data=str(config_path),
    epochs=30,
    imgsz=1280,
    batch=24,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    close_mosaic=10,
    patience=15,
    save_period=5,
    project='runs/finetune',
    name='yolo11s_basketball',
    exist_ok=True,
    plots=True,
    # SportsMOT broadcasts already have heavy variation; modest augs work best.
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    verbose=True,
)

RUN_DIR = Path('runs/finetune/yolo11s_basketball')
BEST    = RUN_DIR / 'weights' / 'best.pt'
print('Final weights at', BEST)

## 4. Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(RUN_DIR / 'results.csv')
df.columns = [c.strip() for c in df.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(df['epoch'], df['train/box_loss'], label='box')
axes[0].plot(df['epoch'], df['train/cls_loss'], label='cls')
axes[0].plot(df['epoch'], df['train/dfl_loss'], label='dfl')
axes[0].set_title('train losses'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(df['epoch'], df['val/box_loss'], label='box')
axes[1].plot(df['epoch'], df['val/cls_loss'], label='cls')
axes[1].plot(df['epoch'], df['val/dfl_loss'], label='dfl')
axes[1].set_title('val losses'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5')
axes[2].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
axes[2].set_title('val mAP'); axes[2].set_xlabel('epoch'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

best_row = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"Best epoch: {int(best_row['epoch'])}  mAP@0.5={best_row['metrics/mAP50(B)']:.3f}  mAP@0.5:0.95={best_row['metrics/mAP50-95(B)']:.3f}")

## 5. Final detection mAP on the val split

In [ ]:
model = YOLO(str(BEST))
metrics = model.val(
    data=str(config_path),
    imgsz=1280,
    batch=16,
    device=0,
)
print(f'mAP@0.5      : {metrics.box.map50:.3f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.3f}')

## 6. Tracking metrics (MOTA / IDF1) on a SportsMOT val sequence

This is the apples-to-apples comparison against the local pretrained
baseline saved in `outputs/baselines/v_BgwzTUxJaeU_c008_pretrained_yolo11n.json`.

We use `scripts/run_baseline.py` with the just-trained `best.pt`. The
raw SportsMOT data needs to be on Drive too (the basketball val
sequence we pulled in notebook 01).

In [ ]:
# Find a basketball val sequence to run the comparison on.
val_split_dir = Path(RAW_ROOT) / 'val'
if not val_split_dir.exists():
    print(f'⚠️  No raw val sequences at {val_split_dir}. Skipping tracking eval.')
    print('   Run notebook 01 to populate it, or pick any local sequence.')
else:
    # Match against basketball.txt to make sure we pick a basketball seq.
    bball = set((Path(RAW_ROOT) / 'splits_txt' / 'basketball.txt').read_text().split())
    candidates = [p for p in val_split_dir.iterdir() if p.is_dir() and p.name in bball]
    assert candidates, 'No basketball val sequences found.'
    eval_seq = candidates[0]
    print('Running tracking eval on', eval_seq)
    !python scripts/run_baseline.py \
        --seq {eval_seq.name} \
        --data-root {RAW_ROOT} \
        --weights {BEST} \
        --tag finetuned_yolo11s \
        --no-class-filter \
        --imgsz 1280

In [ ]:
# Build an updated comparison table including the fine-tuned run.
!python scripts/compare_baselines.py

## 7. Persist `best.pt` to Drive

Copy the trained weights to your mounted Drive so they survive runtime
shutdown and you can download them back to your local machine.

In [ ]:
import shutil
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')
out_pt = Path(WEIGHTS_DRIVE) / f'yolo11s_basketball_{stamp}.pt'
shutil.copy(BEST, out_pt)
print('Saved:', out_pt)

# Also copy the comparison table next to the weights for easy reference.
for src in ('outputs/baselines/COMPARISON.md',):
    s = Path(src)
    if s.exists():
        shutil.copy(s, Path(WEIGHTS_DRIVE) / s.name)

print()
print('Done. To use these weights locally:')
print(f'  1. Download {out_pt.name} from Drive')
print('  2. Run on any clip:')
print('       python scripts/demo_detect.py --source clip.mp4 \\')
print(f'           --weights {out_pt.name} --no-class-filter')
print('  3. Reproduce the tracking baseline:')
print('       python scripts/run_baseline.py \\')
print(f'           --weights {out_pt.name} --tag finetuned_yolo11s --no-class-filter')